# 40 ScienceWorld drainage-aware recheck
CPU only. Reads archived states from runs 37 and 38 and compares with run 39. No installation, Java, model or simulator execution. Writes a new output directory; preserves all original results.

The sewer is included for **every case**, with its initial contents tracked even if moved elsewhere, and any new contents included at comparison time. Its ID is resolved from the initial state rather than hard-coded. This fixes the known drainage omission but does not prove complete physical scope coverage. No final labels are generated.


In [ ]:
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import runpy
Path("/content/40_scienceworld_drainage_recheck_colab.py").write_text('"""Offline drainage-aware projection recheck. Original results preserved; no final labels."""\nimport gzip, json, hashlib, math\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nROOT=Path(\'/content/drive/MyDrive/ADNN_Irreversibility/results\')\nPILOT=ROOT/\'scienceworld_physical_pilot_20261107T091929052223Z\'\nRECOVERY=ROOT/\'scienceworld_recovery_diagnostic_20261107T094243161617Z\'\n\ndef index(tree):\n    nodes={}; parents={}\n    def visit(node,parent=None):\n        if not isinstance(node,dict) or \'uuid\' not in node:raise ValueError(\'Unsupported object schema\')\n        uid=str(node[\'uuid\'])\n        if uid in nodes:raise ValueError(\'Duplicate UUID\')\n        nodes[uid]=node;parents[uid]=parent\n        children=node.get(\'contents\',{})\n        if not isinstance(children,dict):raise ValueError(\'Unsupported contents schema\')\n        for c in children.values():visit(c,uid)\n    visit(tree);return nodes,parents\n\ndef descendants(nodes,roots):\n    result=set()\n    def visit(uid):\n        if uid in result or uid not in nodes:return\n        result.add(uid)\n        for c in nodes[uid].get(\'contents\',{}).values():visit(str(c[\'uuid\']))\n    for uid in roots:visit(uid)\n    return result\n\ndef make_scope(before,ids):\n    nodes,parents=index(before)\n    if not ids or any(i not in nodes for i in ids):raise ValueError(\'Binding objects absent\')\n    # Include the drainage destination in every case, not only those that failed.\n    # Resolve by unique name in the initial state; keep its UUID thereafter.\n    sewers=[uid for uid,node in nodes.items() if node.get(\'name\')==\'sewer\']\n    if len(sewers)!=1:raise ValueError(\'Expected one sewer; drainage scope cannot be established\')\n    ids=set(ids)|set(sewers)\n    core=descendants(nodes,ids)\n    context={parents[i] for i in core if parents[i] is not None}-core\n    return {\'roots\':sorted(ids),\'core\':sorted(core),\'context\':sorted(context),\'drainage_roots\':sewers,\'scope_version\':\'v2_drainage\'}\n\ndef project(tree,scope):\n    nodes,parents=index(tree)\n    ids=set(scope[\'core\'])|set(scope[\'context\'])|descendants(nodes,scope[\'roots\'])\n    result={}\n    for uid in sorted(ids):\n        if uid not in nodes:result[uid]={\'missing\':True};continue\n        n=nodes[uid]\n        result[uid]={\'parent_uuid\':parents[uid],\n          \'properties\':{k:v for k,v in n.items() if k!=\'contents\'},\n          \'children\':sorted(str(c[\'uuid\']) for c in n.get(\'contents\',{}).values())}\n    return result\n\ndef differences(a,b,path=()):\n    if type(a)!=type(b):return [(path,a,b)]\n    if isinstance(a,dict):\n        result=[]\n        for k in sorted(a.keys()|b.keys()):\n            if k not in a or k not in b:result.append((path+(k,),a.get(k),b.get(k)))\n            else:result.extend(differences(a[k],b[k],path+(k,)))\n        return result\n    return [] if a==b else [(path,a,b)]\n\ndef compare(a,b):\n    ds=differences(a,b)\n    def thermal(d):\n        path,x,y=d\n        return path[-2:]==(\'propMaterial\',\'temperatureC\') and all(type(v) in (int,float) and math.isfinite(v) for v in (x,y))\n    return {\'exact_projection_match\':not ds,\'thermal_only_residual\':bool(ds) and all(thermal(d) for d in ds),\n      \'difference_count\':len(ds),\'differences\':[{\'path\':\'/\'.join(p),\'before\':x,\'after\':y} for p,x,y in ds]}\n\ndef main():\n    assert PILOT.is_dir() and RECOVERY.is_dir(),\'Mount Drive; original pilot and recovery directories are required.\'\n    out=ROOT/(\'scienceworld_drainage_recheck_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    rows=[];errors=[];manifest={}\n    previous_path=ROOT/\'scienceworld_projection_smoke_20261107T132422610043Z\'/\'results.json\'\n    assert previous_path.exists(),\'Previous projection results missing; do not substitute another run.\'\n    previous={r[\'id\']:r for r in json.loads(previous_path.read_text())}\n    manifest[str(previous_path)]=hashlib.sha256(previous_path.read_bytes()).hexdigest()\n    for case in sorted(RECOVERY.glob(\'P[0-9][0-9][0-9]\')):\n        cid=case.name\n        try:\n            binding_path=case/\'action_bindings.json\'\n            bindings=json.loads(binding_path.read_text())[\'bindings\']\n            candidates={tuple(str(i) for i in b[\'obj_ids\']) for b in bindings}\n            if len(candidates)!=1:raise ValueError(\'Missing or ambiguous object-ID binding\')\n            ids=next(iter(candidates))\n            paths=list(PILOT.glob(\'task_*/\'+cid+\'/snapshots.json.gz\'));assert len(paths)==1\n            with gzip.open(paths[0],\'rt\') as f:snap=json.load(f)\n            record=json.loads((case/\'summary.json\').read_text())\n            if record[\'candidate\'].startswith(\'pour \') and len(ids)>=2 and ids[0]==ids[1]:\n                rows.append({\'id\':cid,\'status\':\'excluded_self_pour\',\'final_label\':None});continue\n            scope=make_scope(snap[\'before\'][\'tree\'],set(ids));before=project(snap[\'before\'][\'tree\'],scope)\n            row={\'id\':cid,\'candidate\':record[\'candidate\'],\'scope\':scope,\n                 \'candidate_comparison\':compare(before,project(snap[\'after\'][\'tree\'],scope)),\n                 \'recoveries\':[],\'final_label\':None,\'status\':\'diagnostic_only\'}\n            inputs=[binding_path,paths[0],case/\'summary.json\']\n            for file in sorted(case.glob(\'targeted_*.json.gz\')):\n                with gzip.open(file,\'rt\') as f:trial=json.load(f)\n                row[\'recoveries\'].append({\'undo\':trial[\'test\'][\'undo\'],\n                  \'saved_replay_stable\':trial[\'test\'][\'replay_stable\'],\n                  \'comparison\':compare(before,project(trial[\'end\'][\'tree\'],scope))})\n                inputs.append(file)\n            for file in inputs:manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()\n            old=previous.get(cid, {})\n            row[\'previous_recovery_flags\']=[{\'undo\':t[\'undo\'],\n                \'exact\':t[\'comparison\'][\'exact_projection_match\'],\n                \'thermal_only\':t[\'comparison\'][\'thermal_only_residual\']} for t in old.get(\'recoveries\',[])]\n            row[\'thermal_only_flag_changed\']=any(\n                old_t[\'undo\']==new_t[\'undo\'] and old_t[\'thermal_only\']!=new_t[\'comparison\'][\'thermal_only_residual\']\n                for old_t in row[\'previous_recovery_flags\'] for new_t in row[\'recoveries\'])\n            rows.append(row)\n            print(cid,\'recovery comparisons:\',[(t[\'comparison\'][\'exact_projection_match\'],t[\'comparison\'][\'thermal_only_residual\']) for t in row[\'recoveries\']])\n        except Exception as exc:errors.append({\'id\':cid,\'error\':str(exc)})\n    tests=[t for r in rows for t in r.get(\'recoveries\',[])]\n    summary={\'old_cases_reviewed\':len(rows),\'recovery_comparisons\':len(tests),\n       \'exact_projection_matches\':sum(t[\'comparison\'][\'exact_projection_match\'] for t in tests),\n       \'thermal_only_residuals\':sum(t[\'comparison\'][\'thermal_only_residual\'] for t in tests),\n       \'excluded_self_pours\':[r[\'id\'] for r in rows if r[\'status\']==\'excluded_self_pour\'],\n       \'drainage_roots_by_case\':{r[\'id\']:r[\'scope\'][\'drainage_roots\'] for r in rows if \'scope\' in r},\n       \'cases_with_changed_thermal_only_flag\':[r[\'id\'] for r in rows if r.get(\'thermal_only_flag_changed\')],\n       \'scope_revision\':\'v2: sewer and initial contents tracked in every case; new descendants included\',\n       \'errors\':errors,\'final_labels_created\':0,\'scope_coverage_validated\':False,\n       \'partial_labels_require_exhaustive_search_for_exact_recovery\':True,\n       \'fresh_validation_complete\':False,\'saved_to\':str(out)}\n    for name,data in [(\'summary.json\',summary),(\'results.json\',rows),(\'input_hashes.json\',manifest)]:\n        (out/name).write_text(json.dumps(data,indent=2,allow_nan=False))\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    print(\'DRAINAGE-AWARE RECHECK COMPLETE\');print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':main()\n')
_ = runpy.run_path("/content/40_scienceworld_drainage_recheck_colab.py", run_name="__main__")


Send the final DRAINAGE-AWARE RECHECK COMPLETE summary. A changed thermal-only flag is a correction to the diagnostic scope, not a final class label. Fresh validation and complete restricted recovery searches remain pending.
